In [63]:
import pandas as pd
from pandas.tseries.offsets import DateOffset
import plotly.express as px
from datetime import datetime
import pytz
import re

In [64]:
# Load Playstore dataset
apps = pd.read_csv(r"C:\Users\vanda\Downloads\Play Store Data.csv")
apps.head()  

,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver
0,Photo Editor & Candy Camera & Grid & ScrapBook,ART_AND_DESIGN,4.1,159,19M,"10,000+",Free,0,Everyone,Art & Design,"January 7, 2018",1.0.0,4.0.3 and up
1,Coloring book moana,ART_AND_DESIGN,3.9,967,14M,"500,000+",Free,0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up
2,"U Launcher Lite – FREE Live Cool Themes, Hide ...",ART_AND_DESIGN,4.7,87510,8.7M,"5,000,000+",Free,0,Everyone,Art & Design,"August 1, 2018",1.2.4,4.0.3 and up
3,Sketch - Draw & Paint,ART_AND_DESIGN,4.5,215644,25M,"50,000,000+",Free,0,Teen,Art & Design,"June 8, 2018",Varies with device,4.2 and up
4,Pixel Draw - Number Art Coloring Book,ART_AND_DESIGN,4.3,967,2.8M,"100,000+",Free,0,Everyone,Art & Design;Creativity,"June 20, 2018",1.1,4.4 and up


In [65]:
# Data Cleaning
apps["Rating"] = pd.to_numeric(apps["Rating"],errors = "coerce")

In [66]:
apps["Reviews"] = pd.to_numeric(apps["Reviews"],errors = "coerce")

In [67]:
apps["Installs"] =(apps["Installs"].astype(str).str.replace(",","",regex = False).str.replace("+","",regex = False))

In [68]:
apps["Installs"] = pd.to_numeric(apps["Installs"],errors = "coerce")

In [69]:
# Size Conversion
def convert_size(size):
    size = str(size)
    if "M" in size:
        return float(size.replace("M",""))
    elif "k" in size or "K" in size:
        return float(re.sub("[kK]","",size))/1024
    else:
        return None
    

In [70]:
apps["Size_MB"] = apps["Size"].apply(convert_size)

In [71]:
# Date Conversion
apps["Last Updated"] = pd.to_datetime(apps["Last Updated"],errors = "coerce")

In [72]:
# Filtering Data
apps = apps[
    (apps["Rating"] > 4.2) &
    (apps["Category"].str.startswith(("T","P"),na = False)) &
    (apps["Reviews"] > 1000) &
    (~apps["App"].str.contains(r"\d",regex = True,na = False)) &
    (apps["Size_MB"].between(20,80))
    ]
    

In [73]:
# Monthly Aggregation
apps["Month"] = apps["Last Updated"].dt.to_period("M").astype(str)

In [74]:
area_apps = (apps.groupby(["Month","Category"],as_index=False)["Installs"].sum().reset_index())

In [75]:
area_apps["Installs"] = (area_apps.groupby("Category")["Installs"].cumsum())

In [76]:
area_apps["Month"] = pd.to_datetime(area_apps["Month"])

In [77]:
area_apps = area_apps.sort_values("Month")

In [78]:
# Translation
translations = {
    "TRAVEL_AND_LOCAL":"voyage et local", # French
    "PRODUCTIVITY":"productividad", # Spanish
     "PHOTOGRAPHY":"写真撮影" # Japanese
}

In [79]:
area_apps["Category"] = area_apps["Category"].replace(translations)

In [80]:
pivot = area_apps.pivot(index="Month",
                        columns = "Category",
                        values = "Installs").fillna(0)

In [81]:
# Calculate Growth
growth = pivot.pct_change()

In [82]:
highlight_months = growth[(growth>0.25).any(axis=1)].index

In [83]:
# Time Restriction
ist = pytz.timezone("Asia/Kolkata")
current_time = datetime.now(ist).time()

In [84]:
if current_time >= datetime.strptime("16:00","%H:%M").time() and current_time <= datetime.strptime("18:00","%H:%M").time() :

    # Plot Stacked Area Chart
    fig = px.area(
        area_apps,
        x = "Month",
        y = "Installs",
        color = "Category",
        title = "Cumulative Installs Over Time by App Category"
    )
    
    for month in highlight_months:
        fig.add_vrect(
            x0 = month,
            x1 = month + DateOffset(months=1),
            fillcolor = "rgba(255,165,0,0.10)",
            line_width = 0,
            layer = "below",
            opacity = 0.2
        )
        
    fig.update_layout(
            xaxis_title = "Month",
            yaxis_title = "Total Installs",
            hovermode = "x unified",
            width = 1100,
            height = 650,
            legend_title = "App Category"
        )
        
    fig.show()

else:
    print("Stacked Area Chart is available only  between 4:00 PM IST and 6:00 PM IST") 
        

Stacked Area Chart is available only  between 4:00 PM IST and 6:00 PM IST


In [85]:
#CONCLUSION:
#The stacked area chart successfully visualizes the cumulative installs over time for the selected app categories after applying all the required filters, 
#including average rating (≥ 4.2), app names without numbers, categories starting with T or P, reviews greater than 1,000, and app sizes between 20 MB 
#and 80 MB. The legend labels are translated as specified, and months with more than 25% month-over-month growth are highlighted to 
#emphasize significant increases in installs. Additionally, the visualization is configured to be displayed only between 4:00 PM IST and 6:00 PM IST, 
#ensuring all assignment requirements are met.